In [ ]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 35.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import optuna
import random
from torch.optim.swa_utils import AveragedModel, SWALR
import time
import torch.optim as optim

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import os
from datetime import datetime

In [ ]:
torch.backends.cudnn.benchmark = True

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_data = np.load(r"/content/drive/MyDrive/quickdraw_train.npz")
print(train_data.files)

test_data = np.load(r"/content/drive/MyDrive/quickdraw_test.npz")
print(test_data.files)

['x_train', 'y_train', 'class_names']
['test_images']


In [ ]:
X = train_data["x_train"]
y = train_data["y_train"]
class_names = train_data["class_names"]

X_test = test_data["test_images"]

print("Train shape:", X.shape)
print("Labels shape:", y.shape)
print("Test shape:", X_test.shape)
print("Number of classes:", len(class_names))

Train shape: (60000, 784)
Labels shape: (60000,)
Test shape: (15000, 784)
Number of classes: 15


In [ ]:
print("Unique labels:", np.unique(y))

Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]


In [ ]:
print(X.shape)

(60000, 784)


In [ ]:
print(X.dtype)
print(X.min(), X.max())

uint8
0 255


In [ ]:
class QuickDrawDataset(Dataset):
    def __init__(self, images, labels=None, augment=False):
        self.images = images.astype(np.float32) / 255.0
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def augment_fn(self, x):
      x = x.reshape(28, 28)
      # Random horizontal flip
      if random.random() > 0.5:
          x = torch.flip(x, dims=[1])
      # Random erasing
      if random.random() > 0.5:
          i, j = random.randint(0, 20), random.randint(0, 20)
          h, w = random.randint(4, 8), random.randint(4, 8)
          x[i:i+h, j:j+w] = 0
      x = x.reshape(784)
      noise = torch.randn_like(x) * 0.03
      return torch.clamp(x + noise, 0, 1)

    def __getitem__(self, idx):
        x = torch.tensor(self.images[idx])

        if self.augment:
            x = self.augment_fn(x)

        if self.labels is not None:
            y = torch.tensor(self.labels[idx]).long()
            return x, y

        return x

In [ ]:
class ChampionMLP(nn.Module):
    def __init__(self, input_size, num_classes, hidden_sizes, dropout):
        super().__init__()

        layers = []
        prev = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, num_classes))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Mixup
        mixed_x, y_a, y_b, lam = mixup_data(x, y, alpha=0.3)

        optimizer.zero_grad()
        outputs = model(mixed_x)
        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        loss.backward()
        optimizer.step()

        # For accuracy logging, use original x (not mixed)
        with torch.no_grad():
            clean_out = model(x)
        preds = torch.argmax(clean_out, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return accuracy_score(all_labels, all_preds)


def validate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            preds = torch.argmax(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    return accuracy_score(all_labels, all_preds)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
TOTAL_TRIALS = 25
CURRENT_TRIAL = 0

# ── Log file setup ────────────────────────────────────────────
LOG_FILE = f"optuna_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def log(msg):
    print(msg)
    with open(LOG_FILE, "a") as f:
        f.write(msg + "\n")

log(f"🗂 Log file: {LOG_FILE}")
log(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
# ─────────────────────────────────────────────────────────────

def objective(trial):

    global CURRENT_TRIAL
    CURRENT_TRIAL += 1

    log("\n" + "="*60)
    log(f"🔎 Trial {CURRENT_TRIAL}/{TOTAL_TRIALS} STARTED")
    log("="*60)

    n_layers    = trial.suggest_int("n_layers", 4, 6)
    first_layer = trial.suggest_int("first_layer", 768, 1024, step=128)
    shrink      = trial.suggest_float("shrink", 0.55, 0.75)
    dropout     = trial.suggest_float("dropout", 0.15, 0.35)
    lr          = trial.suggest_float("lr", 5e-4, 3e-3, log=True)
    wd          = trial.suggest_float("wd", 1e-6, 1e-3, log=True)
    arch_type   = trial.suggest_categorical("arch_type", ["shrink", "flat"])

    if arch_type == "shrink":
        hidden_sizes = [int(first_layer * (shrink ** i)) for i in range(n_layers)]
    elif arch_type == "flat":
        hidden_sizes = [first_layer] * n_layers

    # ── For 6-layer flat, scale down if too large ─────────────
    if arch_type == "flat" and n_layers == 6:
        est_params = sum(a * b for a, b in zip([784] + hidden_sizes, hidden_sizes + [15]))
        if est_params > 3_000_000:
            # Find largest width that fits
            for w in range(first_layer, 256, -32):
                hidden_sizes = [w] * n_layers
                est = sum(a * b for a, b in zip([784] + hidden_sizes, hidden_sizes + [15]))
                if est <= 3_000_000:
                    log(f"⚙️  6-layer flat scaled down to width {w}")
                    break
    # ─────────────────────────────────────────────────────────

    log(f"Arch Type: {arch_type}")
    log(f"Architecture: {hidden_sizes}")
    log(f"Dropout: {dropout:.3f} | LR: {lr:.5f} | WD: {wd:.6f}")

    SWA_START = 25
    SWA_LR    = lr * 0.3

    cv_scores = []
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

        log(f"\n📂 Fold {fold}/3")

        train_ds = QuickDrawDataset(X[train_idx], y[train_idx], augment=True)
        val_ds   = QuickDrawDataset(X[val_idx],   y[val_idx])

        train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=1024)

        model = ChampionMLP(784, 15, hidden_sizes, dropout).to(device)

        param_count = count_parameters(model)
        log(f"Parameters: {param_count:,}")

        if param_count > 3_000_000:
            log("❌ Exceeds 3M parameter limit — Skipping trial")
            return 0

        swa_model     = AveragedModel(model)
        optimizer     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        scheduler     = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
        swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
        criterion     = nn.CrossEntropyLoss(label_smoothing=0.05)

        best_val   = 0
        patience   = 7
        counter    = 0
        swa_active = False

        for epoch in range(40):

            train_acc = train_epoch(model, train_loader, optimizer, criterion)

            if epoch >= SWA_START:
                swa_model.update_parameters(model)
                swa_scheduler.step()
                swa_active = True
            else:
                scheduler.step()

            if swa_active:
                update_bn(train_loader, swa_model, device=device)
                val_acc = validate(swa_model, val_loader)
            else:
                val_acc = validate(model, val_loader)

            msg = (f"Epoch {epoch+1:02d}/40 | "
                   f"Train Acc: {train_acc:.4f} | "
                   f"Val Acc: {val_acc:.4f}"
                   f"{' [SWA]' if swa_active else ''}")
            log(msg)

            if swa_active:
                if val_acc > best_val:
                    best_val = val_acc
                    counter  = 0
                else:
                    counter += 1
                if counter >= patience:
                    log("⏹ Early stopping triggered")
                    break
            else:
                if val_acc > best_val:
                    best_val = val_acc

        log(f"✅ Fold {fold} Best Val Acc: {best_val:.4f}")
        cv_scores.append(best_val)

    mean_score = np.mean(cv_scores)

    log("\n🏁 Trial Completed")
    log(f"Mean CV Accuracy: {mean_score:.4f}")
    log(f"Trials Remaining: {TOTAL_TRIALS - CURRENT_TRIAL}")
    log("="*60)

    return mean_score

🗂 Log file: optuna_log_20260223_191521.txt
Started at: 2026-02-23 19:15:21


In [1]:
# study = optuna.create_study(direction="maximize")

# print("\n🚀 Starting Hyperparameter Search")
# print(f"Total Trials: {TOTAL_TRIALS}\n")

# study.optimize(objective, n_trials=TOTAL_TRIALS)

# print("\n🎯 Search Completed")
# print("Best Parameters:")
# print(study.best_params)
# print("Best CV Accuracy:", study.best_value)